In [9]:
import os
import streamlit as st
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores  import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv(override=True)

True

In [2]:
llm = ChatGroq(model="llama-3.1-8b-instant", api_key= os.getenv("GROQ_API_KEY"), max_tokens=1000)

In [3]:
loader = PyPDFLoader("Hospital_Policy_Single_Document.pdf")
docs = loader.load()
docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-07T05:50:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-07T05:50:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Hospital_Policy_Single_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Hospital General Policies\nAdmission Policy\n1\nPatients must provide valid ID proof during admission.\n2\nAdvance deposit required before room allocation.\n3\nEmergency admissions allowed without advance; must settle within 24 hours.\n4\nRoom type subject to availability.\nDischarge Policy\n1\nDoctor initiates discharge after clinical clearance.\n2\nBilling clearance required before discharge.\n3\nPharmacy clearance mandatory.\n4\nDischarge summary will be provided.\nHospital Packages\n1\nMaternity Package – Normal delivery (3 days stay).\n2\nC-section Package – 5 days stay.\n3\nGe

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = text_splitter.split_documents(docs)
splitted_docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-07T05:50:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-07T05:50:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Hospital_Policy_Single_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Hospital General Policies\nAdmission Policy\n1\nPatients must provide valid ID proof during admission.\n2\nAdvance deposit required before room allocation.\n3\nEmergency admissions allowed without advance; must settle within 24 hours.\n4\nRoom type subject to availability.\nDischarge Policy\n1\nDoctor initiates discharge after clinical clearance.\n2\nBilling clearance required before discharge.\n3\nPharmacy clearance mandatory.\n4\nDischarge summary will be provided.\nHospital Packages\n1\nMaternity Package – Normal delivery (3 days stay).\n2\nC-section Package – 5 days stay.\n3\nGe

In [5]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small", openai_api_key= os.getenv("OPENAI_API_KEY"))

In [6]:
vectorstore = FAISS.from_documents(splitted_docs, embeddings)

In [7]:
retriever = vectorstore.as_retriever()

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions about the hospital policy."),
    ("user","{input}")
])

In [10]:
chain = prompt | llm | StrOutputParser()

In [11]:
response = chain.invoke("What is the policy on patient confidentiality?")
response

"Our hospital policy on patient confidentiality is based on the Health Insurance Portability and Accountability Act (HIPAA) and other relevant laws and regulations. \n\nAccording to our policy, all hospital staff members are required to maintain the confidentiality of patients' protected health information (PHI). This includes but is not limited to:\n\n1. Verbal communications: Staff members are not allowed to discuss patients' PHI over the phone, in public areas, or in the presence of unauthorized individuals.\n2. Written communications: Staff members must use secure and encrypted methods to communicate PHI, such as using electronic medical records (EMRs) or secure email.\n3. Electronic communications: Staff members are not allowed to share PHI on social media, text messages, or other public forums.\n4. Access to PHI: Only authorized staff members with a legitimate need to access PHI are allowed to view or access patient records. Access is granted through our electronic medical record